# Blue Book Encounter Phenomenology: OCR & Preprocessing Pipeline

Colab Pro optimized. Run on GPU runtime (A100 preferred, T4 workable).

**Pipeline:**
1. Download Blue Book PDFs from Black Vault (targeted subset)
2. Assess existing OCR quality per document
3. Run Marker on documents needing re-OCR
4. Extract and structure witness narratives
5. Build case index with metadata + clean text
6. Save to Google Drive for persistence across sessions

## Cell 1: Environment Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import json

# Project directory on Drive (persists across sessions)
PROJECT_DIR = "/content/drive/MyDrive/blue_book_phenomenology"
PDF_DIR = f"{PROJECT_DIR}/pdfs"
RAW_OCR_DIR = f"{PROJECT_DIR}/ocr_raw"         # Black Vault existing text
MARKER_OUT_DIR = f"{PROJECT_DIR}/marker_output"  # Marker re-OCR'd text
FINAL_CORPUS_DIR = f"{PROJECT_DIR}/corpus"       # Clean structured text
METADATA_DIR = f"{PROJECT_DIR}/metadata"

for d in [PROJECT_DIR, PDF_DIR, RAW_OCR_DIR, MARKER_OUT_DIR,
          FINAL_CORPUS_DIR, METADATA_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project directory structure created on Google Drive.")

## Cell 2: Install Dependencies

In [ ]:
%%bash
# Marker (latest from datalab-to)
pip install marker-pdf -q

# NLP pipeline dependencies
pip install sentence-transformers umap-learn hdbscan bertopic -q
pip install spacy -q
python -m spacy download en_core_web_sm -q

# Utilities
pip install polars pyarrow tqdm requests beautifulsoup4 -q

echo "Dependencies installed."

## Cell 3: GPU Detection & Batch Size Configuration

In [ ]:
import torch

def get_gpu_config():
    """Detect GPU and set optimal Marker batch parameters."""
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Switch to GPU runtime.")

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9

    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")

    # Marker uses ~5GB peak, ~3.5GB average per worker
    # Leave 2GB headroom for system
    max_workers = max(1, int((vram_gb - 2) / 5))

    # batch_multiplier scales internal batch sizes
    # Higher = faster but more VRAM
    if vram_gb >= 35:  # A100
        batch_multiplier = 4
        max_workers = min(max_workers, 6)  # Cap to avoid OOM
    elif vram_gb >= 14:  # T4 / V100
        batch_multiplier = 2
        max_workers = min(max_workers, 2)
    else:
        batch_multiplier = 1
        max_workers = 1

    config = {
        "gpu_name": gpu_name,
        "vram_gb": round(vram_gb, 1),
        "marker_workers": max_workers,
        "batch_multiplier": batch_multiplier,
        "estimated_pages_per_minute": max_workers * 6,
    }

    print(f"\nOptimal config:")
    print(f"  Marker workers: {max_workers}")
    print(f"  Batch multiplier: {batch_multiplier}")
    print(f"  Est. throughput: ~{config['estimated_pages_per_minute']} pages/min")

    return config

gpu_config = get_gpu_config()

## Cell 4: Download Blue Book Case Index from Black Vault

Black Vault organizes Blue Book by microfilm "rolls." Each roll is a large PDF containing multiple cases.

**Strategy:**
- First, download the case index to identify which rolls contain the 701 Unidentified cases
- Then download only those rolls
- This avoids downloading the entire 130K-page archive

**Important:** The 701 Unidentified cases list is widely available. Brad Sparks maintains the most authoritative version: "The Comprehensive Catalog of Project Blue Book Unknowns"

Download it manually and place at:
`/content/drive/MyDrive/blue_book_phenomenology/metadata/bb_unknowns.csv`

See: https://www.nicap.org/bb/BB_Unknowns.pdf

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from tqdm import tqdm

unknowns_path = f"{METADATA_DIR}/bb_unknowns.csv"

if os.path.exists(unknowns_path):
    import polars as pl
    unknowns = pl.read_csv(unknowns_path)
    print(f"Loaded {len(unknowns)} Unidentified cases from index.")
else:
    print("WARNING: Unidentified cases index not found.")
    print(f"Place Brad Sparks' catalog at: {unknowns_path}")
    print("See: https://www.nicap.org/bb/BB_Unknowns.pdf")
    print("Or compile from NICAP's Blue Book archive.")

## Cell 5: Download Target PDFs from Internet Archive

The NARA Project Blue Book microfilm reels are hosted on Internet Archive at:
`https://archive.org/download/nara-pbb/NARA-PBB{roll_number}.pdf`

Each roll is 180-300MB and contains hundreds of pages spanning multiple cases.

**Roll coverage:** Reel 1 = Cases 1-54 (Summer 1947). Rolls 1-10 cover 1947-1952, the peak Unidentified period. 90 rolls total.

In [ ]:
def download_roll(roll_number, output_dir):
    """Download a single Blue Book microfilm roll PDF from Internet Archive."""
    filename = f"NARA-PBB{roll_number}.pdf"
    filepath = os.path.join(output_dir, filename)

    if os.path.exists(filepath):
        print(f"  Roll {roll_number} already downloaded.")
        return filepath

    url = f"https://archive.org/download/nara-pbb/{filename}"

    try:
        resp = requests.get(url, stream=True, timeout=60)
        resp.raise_for_status()
        total_size = int(resp.headers.get('content-length', 0))
        
        with open(filepath, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True,
                      desc=f"Roll {roll_number}", leave=False) as pbar:
                for chunk in resp.iter_content(chunk_size=65536):
                    f.write(chunk)
                    pbar.update(len(chunk))
        return filepath
    except Exception as e:
        print(f"  Failed to download roll {roll_number}: {e}")
        # Clean up partial download
        if os.path.exists(filepath):
            os.remove(filepath)
        return None

# Download first 10 rolls for POC (1947-1952, peak Unidentified period)
POC_ROLLS = range(1, 11)

print("Downloading target rolls from Internet Archive...")
print("(Each roll is 180-300MB, expect ~5-10 min total)\n")
downloaded = []
for roll in POC_ROLLS:
    result = download_roll(roll, PDF_DIR)
    if result:
        downloaded.append(result)

print(f"\nDownloaded {len(downloaded)}/{len(list(POC_ROLLS))} rolls.")

## Cell 6: OCR Quality Assessment

Before running Marker on everything, check if embedded text is usable. Many Blue Book scans have no embedded text (pure image PDFs) or have bad OCR from earlier digitization passes.

**Heuristic:** Extract embedded text with PyMuPDF. If character count per page is < 50 or contains high ratio of garbage characters, flag for Marker re-OCR.

In [ ]:
import fitz  # PyMuPDF

def assess_pdf_ocr_quality(pdf_path, sample_pages=5):
    """
    Quick assessment of existing OCR quality in a PDF.
    Returns: 'good', 'partial', or 'needs_ocr'
    """
    try:
        doc = fitz.open(pdf_path)
    except Exception:
        return 'needs_ocr', 0

    total_pages = len(doc)
    if total_pages == 0:
        return 'needs_ocr', 0

    # Sample evenly spaced pages
    sample_indices = [int(i * total_pages / sample_pages)
                      for i in range(min(sample_pages, total_pages))]

    char_counts = []
    garbage_ratios = []

    for idx in sample_indices:
        page = doc[idx]
        text = page.get_text()
        char_count = len(text.strip())
        char_counts.append(char_count)

        if char_count > 0:
            garbage = sum(1 for c in text if ord(c) > 127 or c in '\u25a1\u25a0\u25cf\u25cb\u25ca\u25aa')
            garbage_ratios.append(garbage / char_count)
        else:
            garbage_ratios.append(1.0)

    doc.close()

    avg_chars = sum(char_counts) / len(char_counts)
    avg_garbage = sum(garbage_ratios) / len(garbage_ratios)

    if avg_chars < 50:
        return 'needs_ocr', total_pages
    elif avg_garbage > 0.1:
        return 'partial', total_pages
    else:
        return 'good', total_pages

print("Assessing OCR quality of downloaded PDFs...")
ocr_assessment = {}
for pdf_path in tqdm(downloaded):
    quality, pages = assess_pdf_ocr_quality(pdf_path)
    ocr_assessment[pdf_path] = {"quality": quality, "pages": pages}

needs_ocr = [p for p, v in ocr_assessment.items() if v["quality"] != "good"]
good_ocr = [p for p, v in ocr_assessment.items() if v["quality"] == "good"]
total_pages_needing_ocr = sum(v["pages"] for p, v in ocr_assessment.items()
                              if v["quality"] != "good")

print(f"\nResults:")
print(f"  Good existing OCR: {len(good_ocr)} PDFs")
print(f"  Needs Marker OCR: {len(needs_ocr)} PDFs ({total_pages_needing_ocr} pages)")
print(f"  Est. Marker time: ~{total_pages_needing_ocr / gpu_config['estimated_pages_per_minute']:.0f} minutes")

# Save assessment for resumption across sessions
with open(f"{METADATA_DIR}/ocr_assessment.json", "w") as f:
    json.dump(ocr_assessment, f, indent=2)

## Cell 7: Run Marker OCR (Batch Processing)

Run Marker on PDFs that need OCR. Force OCR mode since these are scanned military documents from the 1940s-1960s.

**Key flags:**
- `force_ocr`: Force full OCR even if text layer exists (critical for Blue Book since embedded text is often garbled)
- `langs English`: All Blue Book reports are in English
- `disable_image_extraction`: We don't need images, saves time

In [ ]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

def run_marker_batch(pdf_paths, output_dir, force_ocr=True):
    """
    Process a batch of PDFs with Marker.
    Saves markdown output and metadata JSON for each.
    """
    # Load models once (expensive, ~2-3 min)
    print("Loading Marker models...")
    model_dict = create_model_dict()

    converter = PdfConverter(
        artifact_dict=model_dict,
        config={
            "force_ocr": force_ocr,
            "languages": ["English"],
            "disable_image_extraction": True,
            "paginate_output": True,
        }
    )

    results = {}
    for pdf_path in tqdm(pdf_paths, desc="OCR processing"):
        try:
            rendered = converter(pdf_path)
            text, _, images = text_from_rendered(rendered)

            # Save markdown
            basename = os.path.splitext(os.path.basename(pdf_path))[0]
            md_path = os.path.join(output_dir, f"{basename}.md")
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(text)

            results[pdf_path] = {
                "status": "success",
                "output": md_path,
                "char_count": len(text),
            }

        except Exception as e:
            results[pdf_path] = {
                "status": "error",
                "error": str(e),
            }
            print(f"  ERROR on {os.path.basename(pdf_path)}: {e}")

    return results

# Process PDFs needing OCR
if needs_ocr:
    marker_results = run_marker_batch(needs_ocr, MARKER_OUT_DIR, force_ocr=True)

    # Save results for session resumption
    with open(f"{METADATA_DIR}/marker_results.json", "w") as f:
        json.dump(marker_results, f, indent=2)

    successes = sum(1 for v in marker_results.values() if v["status"] == "success")
    print(f"\nMarker OCR complete: {successes}/{len(needs_ocr)} PDFs processed.")
else:
    print("All PDFs have acceptable existing OCR. Skipping Marker.")

# For PDFs with good existing OCR, extract text with PyMuPDF
print("\nExtracting text from good-OCR PDFs...")
for pdf_path in tqdm(good_ocr):
    doc = fitz.open(pdf_path)
    basename = os.path.splitext(os.path.basename(pdf_path))[0]
    md_path = os.path.join(RAW_OCR_DIR, f"{basename}.md")

    pages_text = []
    for page in doc:
        pages_text.append(page.get_text())

    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n\n---\n\n".join(pages_text))

    doc.close()

print("Text extraction complete.")

## Cell 8: Case Segmentation

Each roll PDF contains multiple cases. We need to segment the OCR'd text into individual cases.

Blue Book cases typically begin with:
- AF Form 112 header
- Case number (e.g., "Case 1234" or "#1234")
- Date and location fields

**Strategy:** Use regex patterns to find case boundaries in the OCR'd text, then split into individual case documents.

In [ ]:
import re

def segment_cases(markdown_text, source_roll):
    """
    Split a roll's OCR text into individual cases.
    Returns list of dicts: {case_id, text, approximate_page, source_roll}
    """
    # Common case boundary patterns in Blue Book documents
    case_patterns = [
        r'(?:CASE|Case)\s*#?\s*(\d+)',
        r'(?:REPORT\s+NO|Report\s+No)\.?\s*(\d+)',
        r'(?:AF\s+FORM\s+112)',
        r'(?:PROJECT\s+(?:BLUE\s+BOOK|GRUDGE|SIGN))',
        r'(?:UNIDENTIFIED\s+FLYING\s+OBJECT)',
    ]

    combined_pattern = '|'.join(f'({p})' for p in case_patterns)

    # Find all case boundaries
    boundaries = []
    for match in re.finditer(combined_pattern, markdown_text, re.IGNORECASE):
        boundaries.append(match.start())

    if not boundaries:
        return [{
            "case_id": f"roll_{source_roll}_unsegmented",
            "text": markdown_text,
            "source_roll": source_roll,
        }]

    # Split at boundaries
    cases = []
    for i, start in enumerate(boundaries):
        end = boundaries[i + 1] if i + 1 < len(boundaries) else len(markdown_text)
        case_text = markdown_text[start:end].strip()

        case_num_match = re.search(r'(?:CASE|Case)\s*#?\s*(\d+)', case_text[:500])
        case_id = (case_num_match.group(1) if case_num_match
                   else f"roll_{source_roll}_seg_{i}")

        cases.append({
            "case_id": case_id,
            "text": case_text,
            "char_count": len(case_text),
            "source_roll": source_roll,
        })

    return cases

# Process all OCR'd rolls into individual cases
print("Segmenting rolls into individual cases...")
all_cases = []

for md_dir in [MARKER_OUT_DIR, RAW_OCR_DIR]:
    for md_file in sorted(os.listdir(md_dir)):
        if not md_file.endswith('.md'):
            continue

        md_path = os.path.join(md_dir, md_file)
        with open(md_path, 'r', encoding='utf-8') as f:
            text = f.read()

        roll_num = re.search(r'(\d+)', md_file)
        roll_id = roll_num.group(1) if roll_num else md_file

        cases = segment_cases(text, roll_id)
        all_cases.extend(cases)

print(f"Segmented into {len(all_cases)} individual case documents.")

# Save case corpus
import polars as pl

case_df = pl.DataFrame(all_cases)
case_df.write_parquet(f"{FINAL_CORPUS_DIR}/blue_book_cases.parquet")
print(f"Saved to {FINAL_CORPUS_DIR}/blue_book_cases.parquet")

## Cell 9: Witness Narrative Extraction

Within each case document, extract specifically the witness narrative sections. These are typically in:
- Section 11 of AF Form 112: "Brief summary of sighting"
- Attached witness statements
- Interview transcripts

This is a heuristic extraction -- Marker's layout detection helps but the forms are inconsistent across decades.

In [ ]:
def extract_witness_narrative(case_text):
    """
    Extract the witness narrative portion from a Blue Book case document.
    Returns the narrative text, or the full text if no clear section found.
    """
    narrative_patterns = [
        r'(?:BRIEF\s+SUMMARY\s+OF\s+SIGHTING)(.*?)(?:CONCLUSION|ANALYSIS|COMMENTS)',
        r'(?:DESCRIPTION\s+OF\s+SIGHTING)(.*?)(?:ACTION\s+TAKEN|CONCLUSION)',
        r'(?:NARRATIVE)(.*?)(?:CONCLUSION|ANALYSIS)',
        r'(?:WITNESS\s+STATEMENT)(.*?)(?:SIGNED|INVESTIGATOR)',
        r'(?:STATEMENT\s+OF\s+WITNESS)(.*?)(?:SIGNED|INVESTIGATOR)',
    ]

    for pattern in narrative_patterns:
        match = re.search(pattern, case_text, re.IGNORECASE | re.DOTALL)
        if match:
            narrative = match.group(1).strip()
            if len(narrative) > 100:
                return narrative

    # Fallback: return the longest paragraph-like section
    paragraphs = re.split(r'\n\s*\n', case_text)
    if paragraphs:
        longest = max(paragraphs, key=len)
        if len(longest) > 100:
            return longest.strip()

    return case_text

# Extract narratives
print("Extracting witness narratives...")
case_df = pl.read_parquet(f"{FINAL_CORPUS_DIR}/blue_book_cases.parquet")

narratives = []
for row in case_df.iter_rows(named=True):
    narrative = extract_witness_narrative(row["text"])
    narratives.append(narrative)

case_df = case_df.with_columns(
    pl.Series("witness_narrative", narratives)
)

case_df = case_df.with_columns(
    pl.col("witness_narrative").str.len_chars().alias("narrative_length")
)

# Filter to cases with meaningful narratives
viable = case_df.filter(pl.col("narrative_length") > 200)
print(f"\nViable cases with narratives > 200 chars: {len(viable)}")
print(f"Median narrative length: {viable['narrative_length'].median()} chars")

viable.write_parquet(f"{FINAL_CORPUS_DIR}/blue_book_narratives.parquet")
print(f"Saved to {FINAL_CORPUS_DIR}/blue_book_narratives.parquet")

## Cell 10: Quick Embedding Clustering (Layer 3 -- No Annotation Needed)

Instant-results cell. No annotation required. Encode all narratives with sentence-transformers, reduce with UMAP, cluster with HDBSCAN. See if Unidentified cases cluster differently.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading sentence-transformer model...")
model = SentenceTransformer('all-mpnet-base-v2')

# Load narratives
narratives_df = pl.read_parquet(f"{FINAL_CORPUS_DIR}/blue_book_narratives.parquet")
texts = narratives_df["witness_narrative"].to_list()

print(f"Encoding {len(texts)} narratives...")
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Save embeddings (expensive to recompute)
np.save(f"{FINAL_CORPUS_DIR}/narrative_embeddings.npy", embeddings)
print("Embeddings saved.")

# UMAP reduction
import umap

print("Running UMAP dimensionality reduction...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
coords_2d = reducer.fit_transform(embeddings)

# HDBSCAN clustering
import hdbscan

print("Running HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom'
)
cluster_labels = clusterer.fit_predict(coords_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_pct = (cluster_labels == -1).sum() / len(cluster_labels) * 100
print(f"\nFound {n_clusters} clusters ({noise_pct:.1f}% noise)")

## Cell 11: Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

scatter = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=cluster_labels,
    cmap='Spectral',
    alpha=0.6,
    s=15,
    edgecolors='none'
)

ax.set_title("Blue Book Witness Narratives \u2014 Embedding Space", fontsize=14)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig(f"{PROJECT_DIR}/embedding_clusters.png", dpi=150)
plt.show()

print(f"Plot saved to {PROJECT_DIR}/embedding_clusters.png")

## Cell 12: BERTopic for Cluster Interpretation

In [ ]:
from bertopic import BERTopic

print("Running BERTopic for cluster interpretation...")
topic_model = BERTopic(
    embedding_model=model,
    umap_model=reducer,
    hdbscan_model=clusterer,
    verbose=True
)

topics, probs = topic_model.fit_transform(texts, embeddings)

# Print topic summaries
print("\n" + "="*60)
print("TOPIC SUMMARIES")
print("="*60)
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    topic_info = topic_model.get_topic(topic_id)
    keywords = ", ".join([word for word, _ in topic_info[:8]])
    count = topics.count(topic_id)
    print(f"\nTopic {topic_id} ({count} cases): {keywords}")

# Save topic model
topic_model.save(f"{PROJECT_DIR}/bertopic_model")
print(f"\nTopic model saved to {PROJECT_DIR}/bertopic_model")

## Cell 13: Session Save & Resume Support

Save all state so you can resume in a new Colab session. Everything is on Google Drive already, but this creates a single checkpoint file for quick state recovery.

In [ ]:
checkpoint = {
    "gpu_config": gpu_config,
    "total_cases_extracted": len(all_cases) if 'all_cases' in dir() else 0,
    "viable_narratives": len(viable) if 'viable' in dir() else 0,
    "n_clusters": n_clusters if 'n_clusters' in dir() else 0,
    "embeddings_path": f"{FINAL_CORPUS_DIR}/narrative_embeddings.npy",
    "narratives_path": f"{FINAL_CORPUS_DIR}/blue_book_narratives.parquet",
    "topic_model_path": f"{PROJECT_DIR}/bertopic_model",
    "status": "complete_through_clustering",
}

with open(f"{PROJECT_DIR}/checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=2)

print("Session checkpoint saved.")
print(f"\nTo resume in a new session:")
print(f"  1. Mount Google Drive")
print(f"  2. Load checkpoint from {PROJECT_DIR}/checkpoint.json")
print(f"  3. Load embeddings from {FINAL_CORPUS_DIR}/narrative_embeddings.npy")
print(f"  4. Load narratives from {FINAL_CORPUS_DIR}/blue_book_narratives.parquet")

## Next Steps (run in subsequent sessions)

1. **ANNOTATION (Layer 1):**
   - Export 100-150 narratives to Label Studio format
   - Annotate phenomenological spans (INITIAL_AWARENESS, etc.)
   - Fine-tune DeBERTa-v3-base for span classification

2. **AFFECT TRAJECTORIES (Layer 2):**
   - `pip install nrclex` or use GoEmotions model
   - Extract sentence-level VAD scores across each narrative
   - Compare trajectories between clusters

3. **MATCHED CONTROL ANALYSIS:**
   - Tag cases as Unidentified (701) vs Identified
   - Overlay USAF classification on embedding plot
   - Statistical tests: are Unidentified cases phenomenologically distinct from Identified cases in embedding space?

4. **CROSS-CORPUS TRANSFER:**
   - Apply the same pipeline to MUFON data, Colares reports, etc.
   - Test whether Blue Book phenomenological signatures generalize